# DyPE → Modular Diffusers — parity, publish, and yarn-vs-spectral

DyPE as a community modular pipeline on **FLUX.1-Krea-dev**. Order: prove the modular block reproduces the
validated DyPE integration → publish PRIVATE to `remyxai/dype-flux-modular` → load via `trust_remote_code`
→ show **yarn vs spectral** at 4K (SEGA's speckle suppression).

**Runtime:** A100 · `HUGGINGFACE_TOKEN` · accept the **FLUX.1-Krea-dev** license · **upload `block.py`**.

## 1 · Install

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf torchvision

## 2 · GPU + HF auth

In [ ]:
import torch
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()

## 3 · Load `block.py` (upload it first)

In [ ]:
import os, importlib
assert os.path.exists("block.py"), "Upload block.py, then re-run."
import block as B; importlib.reload(B)
KREA, DEV, DT = "black-forest-labs/FLUX.1-Krea-dev", "cuda", torch.bfloat16
print("loaded:", [n for n in dir(B) if n in ("DyPEBlock","_install_dype","_DyPEPosEmbed")])

## 4 · Milestone A — PARITY: modular block == DyPE-on-FluxPipeline (same math, both integrations)
Reference = stock FluxPipeline with our `_install_dype` + mu-cap; candidate = the `DyPEBlock` modular pipeline.
Same seed/config at 2048² (DyPE active >1024²). Δ≈0 means the modular denoise integration is faithful.

In [ ]:
import gc, torch
from diffusers import FluxPipeline
RES, STEPS, SEED, GUID = 2048, 20, 0, 4.5
PROMPT = "a photograph of a mountain lake at dawn, pine forest"

# reference: FluxPipeline + our DyPE install + mu-cap
stock = FluxPipeline.from_pretrained(KREA, torch_dtype=DT).to(DEV); stock.vae.enable_tiling()
ms = stock.scheduler.config.get("max_shift", 1.15)
stock.scheduler.register_to_config(base_shift=ms, max_shift=ms)          # DyPE mu-cap
orig_pe, handle = B._install_dype(stock.transformer, method="yarn")
g = torch.Generator(DEV).manual_seed(SEED)
ref = stock(PROMPT, height=RES, width=RES, num_inference_steps=STEPS, guidance_scale=GUID, generator=g, output_type="latent").images.float().cpu()
handle.remove(); stock.transformer.pos_embed = orig_pe
del stock; gc.collect(); torch.cuda.empty_cache()

# candidate: the modular DyPEBlock
pipe = B.DyPEBlock().init_pipeline(); pipe.load_components(dtype=DT); pipe.to(DEV)
g = torch.Generator(DEV).manual_seed(SEED)
out = pipe(prompt=PROMPT, height=RES, width=RES, num_inference_steps=STEPS, guidance_scale=GUID, generator=g, method="yarn", output_type="latent")
cand = (out.images if hasattr(out, "images") else out).float().cpu()
d = (ref - cand).abs().max().item()
print(f"[MILESTONE A] DyPE modular-vs-hook max|Δ| = {d:.3e}  ->  {'PASS' if d < 5e-2 else 'FAIL'}")

## 5 · Milestone C — publish PRIVATE (code + configs only)

In [ ]:
import json
from huggingface_hub import HfApi
REPO_ID, FLUX = "remyxai/dype-flux-modular", "black-forest-labs/FLUX.1-Krea-dev"
open("modular_config.json","w").write(json.dumps(
    {"_class_name":"DyPEBlock","_diffusers_version":"0.41.0.dev0",
     "auto_map":{"ModularPipelineBlocks":"block.DyPEBlock"}}, indent=2))
def comp(sub,lib,cls): return [None,None,{"pretrained_model_name_or_path":FLUX,"revision":None,"subfolder":sub,"type_hint":[lib,cls],"variant":None}]
open("modular_model_index.json","w").write(json.dumps(
    {"_blocks_class_name":"DyPEBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
     "text_encoder":comp("text_encoder","transformers","CLIPTextModel"),
     "tokenizer":comp("tokenizer","transformers","CLIPTokenizer"),
     "text_encoder_2":comp("text_encoder_2","transformers","T5EncoderModel"),
     "tokenizer_2":comp("tokenizer_2","transformers","T5TokenizerFast"),
     "transformer":comp("transformer","diffusers","FluxTransformer2DModel"),
     "vae":comp("vae","diffusers","AutoencoderKL"),
     "scheduler":comp("scheduler","diffusers","FlowMatchEulerDiscreteScheduler")}, indent=2))
api = HfApi(); api.create_repo(REPO_ID, private=True, repo_type="model", exist_ok=True)
for f in ("block.py","modular_config.json","modular_model_index.json"):   # README managed separately
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO_ID)
print("published:", api.list_repo_files(REPO_ID))

## 6 · Milestone D — load from the Hub + run

In [ ]:
import gc, torch
for v in ("pipe","stock"):
    globals().pop(v, None)
gc.collect(); torch.cuda.empty_cache()
from diffusers import ModularPipeline
hub = ModularPipeline.from_pretrained("remyxai/dype-flux-modular", trust_remote_code=True)
print("loaded block:", type(hub.blocks).__name__)   # expect DyPEBlock
hub.load_components(dtype=DT); hub.to(DEV)
from IPython.display import display
g = torch.Generator(DEV).manual_seed(0)
img = hub(prompt="a photograph of a mountain lake at dawn", height=1024, width=1024, method="yarn").images[0]
img.save("dype_hub_1k.png"); print("OK -> dype_hub_1k.png"); display(img.resize((512,512)))

## 7 · Milestone E — yarn vs spectral @ 4K (SEGA speckle suppression) + peak VRAM

In [ ]:
import gc, torch
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
PROMPT = "a clear blue sky over a calm turquoise sea, a lone red sailboat"   # flat regions = where speckle shows
imgs = {}
for method in ("yarn", "spectral"):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    g = torch.Generator(DEV).manual_seed(0)
    im = hub(prompt=PROMPT, height=4096, width=4096, guidance_scale=4.5, method=method).images[0]
    im.save(f"dype_4k_{method}.png"); imgs[method] = im
    print(f"[{method}] peak reserved = {torch.cuda.max_memory_reserved()/1e9:.1f} GB -> dype_4k_{method}.png")
S=1024; c=Image.new("RGB",(S*2+60,S+80),"white"); d=ImageDraw.Draw(c)
try: font=ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",30)
except Exception: font=ImageFont.load_default()
c.paste(imgs["yarn"].resize((S,S)),(20,60)); c.paste(imgs["spectral"].resize((S,S)),(S+40,60))
d.text((20+S//2-90,20),"DyPE (yarn)",fill="black",font=font); d.text((S+40+S//2-140,20),"DyPE + SEGA (spectral)",fill="black",font=font)
c.save("dype_yarn_vs_spectral_4k.png"); display(c)

## 8 · Next
A PASS + a clean `loaded block: DyPEBlock` + coherent 4K (and less speckle in spectral) = ready to flip public and comment on #14520 with the artifacts.